In [1]:
# --- Configuración de entorno ---

# Añade el directorio raíz al path para que Python encuentre tus módulos
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../..")))  # Sube dos niveles hasta /src

# Importa configuraciones y librerías globales
from config import *
from utils import *

# Carga de los ficheros con la distribución por sexo y actividad

Se han estructurado los datos en la carpeta de inputs para la dimensión socioeconómica de forma que se dispone de un fichero CSV a nivel
de provincia. Así pues, se han definido una función en las utilidades que se encargan de cargar y dar una limpieza inicial a los datos.

Los datos se obtienen del censo anual de población:
https://www.ine.es/dynt3/inebase/index.htm?padre=10607&capsel=10609

In [2]:
path = os.path.join(DATA_INPUTS_DS, "Poblacion mayor 16 por sexo y actividad")

por_sexo_y_actividad = carga_datos_ine(path)

# Veo una muestra de su estructura y contenido
print(por_sexo_y_actividad.info())
por_sexo_y_actividad.sample(5)

<class 'pandas.core.frame.DataFrame'>
Index: 191376 entries, 108 to 313253
Data columns (total 8 columns):
 #   Column                     Non-Null Count   Dtype 
---  ------                     --------------   ----- 
 0   Provincias                 191376 non-null  object
 1   Municipios                 191376 non-null  object
 2   Secciones                  191376 non-null  object
 3   Sexo                       191376 non-null  object
 4   Relación con la actividad  191376 non-null  object
 5   Periodo                    191376 non-null  int64 
 6   Total                      174762 non-null  object
 7   Provincia                  191376 non-null  object
dtypes: int64(1), object(7)
memory usage: 13.1+ MB
None


,Provincias,Municipios,Secciones,Sexo,Relación con la actividad,Periodo,Total,Provincia
92935,24 León,24089 León,2408902001 León sección 02001,Total,Total,2022,1.053,Leon
138396,34 Palencia,34192 Valde-Ucieza,3419201001 Valde-Ucieza sección 01001,Mujeres,Otra situación de inactividad,2023,10,Palencia
129914,34 Palencia,34120 Palencia,3412004006 Palencia sección 04006,Mujeres,Parado/a,2021,31,Palencia
257044,47 Valladolid,47131 Rábano,4713101001 Rábano sección 01001,Total,Ocupado/a,2022,61,Valladolid
234837,42 Soria,42173 Soria,4217302008 Soria sección 02008,Mujeres,"Perceptor/a pensión de incapacidad, jubilación...",2023,109,Soria


In [3]:
hombres = por_sexo_y_actividad[
    (por_sexo_y_actividad["Sexo"] == "Hombres") &
    (por_sexo_y_actividad["Relación con la actividad"] != "Total")
].copy()

mujeres = por_sexo_y_actividad[
    (por_sexo_y_actividad["Sexo"] == "Mujeres") &
    (por_sexo_y_actividad["Relación con la actividad"] != "Total")
].copy()

hombres["Relación con la actividad"] = "Hombres " + hombres["Relación con la actividad"]
mujeres["Relación con la actividad"] = "Mujeres " + mujeres["Relación con la actividad"]

# Estandarización del dataframe de datos del INE

Como se puede observar, el fichero csv de datos del INE tiene un formato poco amigable para el tratamiento de los datos. En lugar de tener una fila
por cada par sección-año y varias columnas (una por factor), tiene múltiples filas con distintos indicadores para una misma sección, lo que resulta
complejo de tratar. Además, se observa como se mezcla el código del municipio, distrito y seccion con el texto, y deberían tener una columna con
los códigos.

In [4]:
hombres_estandarizado = estandarizar_df_ine(hombres, "Relación con la actividad")
print(hombres_estandarizado.info())
hombres_estandarizado.sample(5)

<class 'pandas.core.frame.DataFrame'>
Index: 53160 entries, 129 to 313235
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   Provincia  53160 non-null  object
 1   CMuni      53160 non-null  object
 2   CUSEC      53160 non-null  object
 3   Indicador  53160 non-null  object
 4   Periodo    53160 non-null  int64 
 5   Total      48545 non-null  object
dtypes: int64(1), object(5)
memory usage: 2.8+ MB
None


,Provincia,CMuni,CUSEC,Indicador,Periodo,Total
65419,Burgos,09287,0928701001,Hombres Parado/a,2022,3
111477,Leon,24180,2418001001,Hombres Ocupado/a,2023,302
197992,Segovia,40062,4006201001,"Hombres Perceptor/a pensión de incapacidad, ju...",2022,18
130283,Palencia,34120,3412005004,Hombres Estudiante,2021,53
165323,Salamanca,37202,3720201001,"Hombres Perceptor/a pensión de incapacidad, ju...",2021,81


In [5]:
mujeres_estandarizado = estandarizar_df_ine(mujeres, "Relación con la actividad")
print(mujeres_estandarizado.info())
mujeres_estandarizado.sample(5)

<class 'pandas.core.frame.DataFrame'>
Index: 53160 entries, 147 to 313253
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   Provincia  53160 non-null  object
 1   CMuni      53160 non-null  object
 2   CUSEC      53160 non-null  object
 3   Indicador  53160 non-null  object
 4   Periodo    53160 non-null  int64 
 5   Total      48545 non-null  object
dtypes: int64(1), object(5)
memory usage: 2.8+ MB
None


,Provincia,CMuni,CUSEC,Indicador,Periodo,Total
210973,Segovia,40191,4019101001,Mujeres Otra situación de inactividad,2022,29
168685,Salamanca,37236,3723601001,Mujeres Parado/a,2022,1
144924,Salamanca,37014,3701401001,Mujeres Parado/a,2023,23
222423,Soria,42042,4204201001,Mujeres Estudiante,2023,2
25857,Avila,05229,0522901001,"Mujeres Perceptor/a pensión de incapacidad, ju...",2023,9


In [6]:
# Concatenamos ambos dfs ya que ahora no comparten columnas
por_sexo_y_actividad_estandarizado = pd.concat(
    [hombres_estandarizado, 
     mujeres_estandarizado
    ],
    ignore_index=True
)
print(por_sexo_y_actividad_estandarizado.info())
por_sexo_y_actividad_estandarizado.sample(5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 106320 entries, 0 to 106319
Data columns (total 6 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   Provincia  106320 non-null  object
 1   CMuni      106320 non-null  object
 2   CUSEC      106320 non-null  object
 3   Indicador  106320 non-null  object
 4   Periodo    106320 non-null  int64 
 5   Total      97090 non-null   object
dtypes: int64(1), object(5)
memory usage: 4.9+ MB
None


,Provincia,CMuni,CUSEC,Indicador,Periodo,Total
103805,Zamora,49149,4914901001,Mujeres Parado/a,2021,20
100634,Valladolid,47186,4718611035,Mujeres Estudiante,2021,65
7410,Burgos,09059,0905909003,Hombres Ocupado/a,2023,248
63626,Burgos,09227,0922701001,Mujeres Otra situación de inactividad,2021,14
61459,Burgos,09078,0907801001,Mujeres Parado/a,2022,NaN


## Filtrado de años
Revisando la documentación del INE, en el año 2021 se cambió radicalemente la metodología que define las secciones censales,
y en concreto en Castilla y León se aumentó el numero de censos de 2700 a unos 3500 apróximadamente. Es por ello que, si bien
se dispone de datos de años anteriores, sería complejo y peligroso fragmentar y proyectar los censos de años previos en la malla 
censal actual, por lo que se filtraran datos de años previos

In [7]:
# Reviso los indicadores disponibles
revisar_indicadores_disponibles(por_sexo_y_actividad_estandarizado)

# Filtro por los años 2021 - 2023.
por_sexo_y_actividad_recientes = por_sexo_y_actividad_estandarizado[
    por_sexo_y_actividad_estandarizado["Periodo"].isin([2021, 2022, 2023])
].copy()

📅 Años disponibles:
[2023 2022 2021]
------------------------------------------------------------
🧩 Indicadores demográficos disponibles:
  - Hombres Ocupado/a
  - Hombres Parado/a
  - Hombres Perceptor/a pensión de incapacidad, jubilación, prejubilación
  - Hombres Otra situación de inactividad
  - Hombres Estudiante
  - Mujeres Ocupado/a
  - Mujeres Parado/a
  - Mujeres Perceptor/a pensión de incapacidad, jubilación, prejubilación
  - Mujeres Otra situación de inactividad
  - Mujeres Estudiante
------------------------------------------------------------


In [8]:
# Pivoto los indicadores para tener una columna por indicador y reducir las filas de la tabla
por_sexo_y_actividad_por_seccion = pivotar_indicadores(por_sexo_y_actividad_recientes)
print(por_sexo_y_actividad_por_seccion.info())
por_sexo_y_actividad_por_seccion.sample(5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9709 entries, 0 to 9708
Data columns (total 14 columns):
 #   Column                                                                 Non-Null Count  Dtype 
---  ------                                                                 --------------  ----- 
 0   Provincia                                                              9709 non-null   object
 1   CMuni                                                                  9709 non-null   object
 2   CUSEC                                                                  9709 non-null   object
 3   Periodo                                                                9709 non-null   int64 
 4   Hombres_Estudiante                                                     9709 non-null   object
 5   Hombres_Ocupado_a                                                      9709 non-null   object
 6   Hombres_Otra_situación_de_inactividad                                  9709 non-null   object
 7

,Provincia,CMuni,CUSEC,Periodo,Hombres_Estudiante,Hombres_Ocupado_a,Hombres_Otra_situación_de_inactividad,Hombres_Parado_a,"Hombres_Perceptor_a_pensión_de_incapacidad,_jubilación,_prejubilación",Mujeres_Estudiante,Mujeres_Ocupado_a,Mujeres_Otra_situación_de_inactividad,Mujeres_Parado_a,"Mujeres_Perceptor_a_pensión_de_incapacidad,_jubilación,_prejubilación"
105,Avila,05019,0501904009,2021,57,333,38,44,187,60,331,182,72,92
6512,Segovia,40194,4019403008,2023,28,181,46,17,79,27,210,102,13,84
5995,Segovia,40008,4000801001,2022,1,18,2,1,8,0,9,7,1,6
1097,Burgos,09059,0905905005,2022,24,171,39,19,94,14,179,96,29,102
2644,Leon,24073,2407301001,2023,5,105,8,9,99,8,71,38,10,93


In [9]:
# Creamos totales
cols_num = [
    "Hombres_Estudiante", "Mujeres_Estudiante",
    "Hombres_Ocupado_a", "Mujeres_Ocupado_a",
    "Hombres_Otra_situación_de_inactividad", "Mujeres_Otra_situación_de_inactividad",
    "Hombres_Parado_a", "Mujeres_Parado_a",
    "Hombres_Perceptor_a_pensión_de_incapacidad,_jubilación,_prejubilación",
    "Mujeres_Perceptor_a_pensión_de_incapacidad,_jubilación,_prejubilación"
]

for c in cols_num:
    por_sexo_y_actividad_por_seccion[c] = pd.to_numeric(
        por_sexo_y_actividad_por_seccion[c], errors="coerce"
    )

por_sexo_y_actividad_por_seccion["Total_Estudiante"] = (
    por_sexo_y_actividad_por_seccion["Hombres_Estudiante"] +
    por_sexo_y_actividad_por_seccion["Mujeres_Estudiante"]
)

por_sexo_y_actividad_por_seccion["Total_Ocupado"] = (
    por_sexo_y_actividad_por_seccion["Hombres_Ocupado_a"] +
    por_sexo_y_actividad_por_seccion["Mujeres_Ocupado_a"]
)

por_sexo_y_actividad_por_seccion["Total_Otra_inactividad"] = (
    por_sexo_y_actividad_por_seccion["Hombres_Otra_situación_de_inactividad"] +
    por_sexo_y_actividad_por_seccion["Mujeres_Otra_situación_de_inactividad"]
)

por_sexo_y_actividad_por_seccion["Total_Parado"] = (
    por_sexo_y_actividad_por_seccion["Hombres_Parado_a"] +
    por_sexo_y_actividad_por_seccion["Mujeres_Parado_a"]
)

por_sexo_y_actividad_por_seccion["Total_Pensiones"] = (
    por_sexo_y_actividad_por_seccion["Hombres_Perceptor_a_pensión_de_incapacidad,_jubilación,_prejubilación"] +
    por_sexo_y_actividad_por_seccion["Mujeres_Perceptor_a_pensión_de_incapacidad,_jubilación,_prejubilación"]
)

print(por_sexo_y_actividad_por_seccion.info())
por_sexo_y_actividad_por_seccion.sample(5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9709 entries, 0 to 9708
Data columns (total 19 columns):
 #   Column                                                                 Non-Null Count  Dtype 
---  ------                                                                 --------------  ----- 
 0   Provincia                                                              9709 non-null   object
 1   CMuni                                                                  9709 non-null   object
 2   CUSEC                                                                  9709 non-null   object
 3   Periodo                                                                9709 non-null   int64 
 4   Hombres_Estudiante                                                     9709 non-null   int64 
 5   Hombres_Ocupado_a                                                      9709 non-null   int64 
 6   Hombres_Otra_situación_de_inactividad                                  9709 non-null   int64 
 7

,Provincia,CMuni,CUSEC,Periodo,Hombres_Estudiante,Hombres_Ocupado_a,Hombres_Otra_situación_de_inactividad,Hombres_Parado_a,"Hombres_Perceptor_a_pensión_de_incapacidad,_jubilación,_prejubilación",Mujeres_Estudiante,Mujeres_Ocupado_a,Mujeres_Otra_situación_de_inactividad,Mujeres_Parado_a,"Mujeres_Perceptor_a_pensión_de_incapacidad,_jubilación,_prejubilación",Total_Estudiante,Total_Ocupado,Total_Otra_inactividad,Total_Parado,Total_Pensiones
1890,Burgos,09246,0924601001,2022,19,169,22,12,109,20,110,110,8,67,39,279,132,20,176
9350,Zamora,49200,4920001001,2022,15,239,32,30,176,14,177,167,51,65,29,416,199,81,241
4201,Palencia,34127,3412701001,2022,1,24,3,0,21,2,13,26,2,10,3,37,29,2,31
4969,Salamanca,37169,3716901001,2022,6,48,5,6,40,4,25,32,9,7,10,73,37,15,47
296,Avila,05060,0506001001,2022,5,26,0,1,21,1,22,13,4,7,6,48,13,5,28


In [10]:
# Creamos la carpeta si no existe
os.makedirs(DATA_OUTPUTS_DS, exist_ok=True)

# Rutas de salida
ruta_seccion = os.path.join(DATA_OUTPUTS_DS, "actividad_por_sexo_por_seccion.csv")

# Guardar DataFrames
por_sexo_y_actividad_por_seccion.to_csv(
    ruta_seccion,
    index=False,
    encoding="utf-8-sig",
    sep=";",          # separador de columnas compatible con Excel español
    decimal=",",      # separador decimal europeo
    float_format="%.3f"
)

print(f"✅ Archivos guardados correctamente en: {DATA_OUTPUTS_DS}")

✅ Archivos guardados correctamente en: D:\MASTER EN CIENCIA DE DATOS\TFM\TrabajoFinal\ivst-tfm\data\outputs\DS_Dim_socioeconomica
